# Chad Consulting - Анализ ЦА

**Источники:**
1. Wikipedia: статический парсинг по населению городов РФ
2. World Bank API: доля проникновения интернета по возрастам
3. Mall Customers: открытый датасет на github, сегментация покупателей по доходу в рублях
4. Stepik API: реальные цены русскоязычных онлайн-курсов в рублях
5. Росстат через Wikipedia: статический парсинг средней и медианной зарплаты в РФ
6. Публичные исследования: Dove, Mental Health Foundation для анализа клиентских болей

## 1. Настройки и палитра

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'figures'
for _folder in (DATA_RAW, DATA_PROCESSED, FIGURES):
    _folder.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
CITIES = ['Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань', 'Краснодар', 'Нижний Новгород', 'Ростов-на-Дону']
PALETTE = ['#FF7A00', '#FF4D00', '#FFB300', '#C73E00', '#FF9100', '#E8360C', '#FFC400']
ACCENT_COLOR = '#FF820D'
SEQ_COLORS = ['#FFD199', '#FFB347', '#FF8C00', '#FF6A00', '#E8480C', '#B23A00']
TABLE_HEADER = '#FF6A00'
TABLE_ROW_ALT = '#FFE3C7'
GENDER_COLORS = {'Женский': '#FF6FA5', 'Мужской': '#FF7A00'}
TOPICS = ['Стиль и гардероб', 'Уход за кожей', 'Базовый луксмаксинг', 'Уход за волосами', 'Уверенность в себе']

## 2. Источники данных

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import pandas as pd
class BaseSource(ABC):
    cache_name = 'source.csv'
    def __init__(self, cache_dir):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
    @property
    def cache_path(self):
        return self.cache_dir / self.cache_name
    @abstractmethod
    def fetch(self):
        raise NotImplementedError
    def load(self, refresh=False):
        if not refresh and self.cache_path.exists():
            return pd.read_csv(self.cache_path)
        try:
            df = self.fetch()
        except Exception as err:
            if self.cache_path.exists():
                print(f'[{self.__class__.__name__}] не удалось обновить ({err}), беру кэш')
                return pd.read_csv(self.cache_path)
            raise
        self.save(df)
        return df
    def save(self, df):
        df.to_csv(self.cache_path, index=False)
        print(f'[{self.__class__.__name__}] сохранено строк: {len(df)} -> {self.cache_path.name}')

In [ ]:
import re
from io import StringIO
import pandas as pd
import requests
from bs4 import BeautifulSoup
URL = 'https://ru.wikipedia.org/wiki/Список_городов_России_с_населением_более_100_тысяч_жителей'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
class WikipediaCitiesParser(BaseSource):
    cache_name = 'cities.csv'
    def fetch(self):
        html = requests.get(URL, headers=HEADERS, timeout=20).text
        table = self._pick_table(html)
        return self._tidy(table)
    @staticmethod
    def _pick_table(html):
        soup = BeautifulSoup(html, 'lxml')
        wikitables = soup.select('table.wikitable')
        for tbl in wikitables:
            df = pd.read_html(StringIO(str(tbl)))[0]
            if df.shape[0] > 50:
                return df
        raise RuntimeError('не нашёл таблицу городов на странице')
    def _tidy(self, df):
        df.columns = [' '.join((str(c) for c in col)) if isinstance(col, tuple) else str(col) for col in df.columns]
        city_col = next((c for c in df.columns if 'Город' in c))
        year_cols = [(c, int(m.group())) for c in df.columns if (m := re.search('(19|20)\\d{2}', c))]
        latest_col, latest_year = max(year_cols, key=lambda x: x[1])
        out = pd.DataFrame({'city': df[city_col].astype(str).str.replace('\\[.*?\\]', '', regex=True).str.strip(), 'population_k': pd.to_numeric(df[latest_col], errors='coerce')})
        out = out.dropna(subset=['population_k'])
        out = out[~out['city'].str.contains('итог|город', case=False, na=False)]
        out['population'] = (out['population_k'] * 1000).astype(int)
        out['year'] = latest_year
        return out[['city', 'population', 'year']].sort_values('population', ascending=False).reset_index(drop=True)

In [ ]:
#Берем макропоказатели России по данным World Bank
import time
import pandas as pd
import requests
API = 'https://api.worldbank.org/v2/country/RUS/indicator/{code}?format=json&per_page=100&date=2010:2024'
INDICATORS = {'Население': 'SP.POP.TOTL', 'Доля 15-64 лет, %': 'SP.POP.1564.TO.ZS', 'Интернет-пользователи, %': 'IT.NET.USER.ZS', 'Городское население, %': 'SP.URB.TOTL.IN.ZS'}
class WorldBankSource(BaseSource):
    cache_name = 'worldbank.csv'
    def fetch(self):
        frames = []
        with requests.Session() as session:
            for name, code in INDICATORS.items():
                payload = self._get(session, API.format(code=code))
                rows = payload[1] if len(payload) > 1 and payload[1] else []
                for row in rows:
                    if row['value'] is not None:
                        frames.append({'indicator': name, 'year': int(row['date']), 'value': row['value']})
        if not frames:
            raise RuntimeError('World Bank не вернул данных')
        return pd.DataFrame(frames).sort_values(['indicator', 'year']).reset_index(drop=True)
    @staticmethod
    def _get(session, url, attempts=3, timeout=40):
        last_err = None
        for i in range(attempts):
            try:
                return session.get(url, timeout=timeout).json()
            except requests.RequestException as err:
                last_err = err
                time.sleep(2 * (i + 1))
        raise RuntimeError(f'World Bank недоступен после {attempts} попыток: {last_err}')
    @staticmethod
    def latest(df):
        out = {}
        for name, group in df.groupby('indicator'):
            row = group.sort_values('year').iloc[-1]
            out[name] = {'value': row['value'], 'year': int(row['year'])}
        return out

In [ ]:
import re
from io import StringIO
import pandas as pd
import requests
URL = 'https://ru.wikipedia.org/wiki/Средняя_заработная_плата_в_России'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
class RosstatSalarySource(BaseSource):
    cache_name = 'rosstat_salary.csv'
    def fetch(self):
        html = requests.get(URL, timeout=20, headers=HEADERS).text
        tables = pd.read_html(StringIO(html))
        target = next((t for t in tables if any(('Средняя заработная плата' in ' '.join((str(v) for v in row.values)) for _, row in t.iterrows()))), None)
        if target is None:
            raise RuntimeError('на странице Росстата не нашли таблицу со средней зарплатой')
        cols = [str(c[-1]) if isinstance(c, tuple) else str(c) for c in target.columns]
        year_cols = [(i, int(c)) for i, c in enumerate(cols) if re.fullmatch('\\d{4}', c)]
        mean_row = next((row for _, row in target.iterrows() if 'Средняя' in str(row.iloc[0])))
        median_row = next((row for _, row in target.iterrows() if 'Медианная' in str(row.iloc[0])))
        rows = []
        for idx, year in year_cols:
            m = self._num(mean_row.iloc[idx])
            med = self._num(median_row.iloc[idx])
            if m is None and med is None:
                continue
            rows.append({'year': year, 'mean_rub': m, 'median_rub': med})
        if not rows:
            raise RuntimeError('не удалось распарсить значения зарплат')
        return pd.DataFrame(rows).sort_values('year').reset_index(drop=True)
    @staticmethod
    def _num(cell):
        if cell is None or (isinstance(cell, float) and pd.isna(cell)):
            return None
        m = re.search('\\d[\\d\\s\xa0]*', str(cell).replace(',', '.'))
        if not m:
            return None
        return int(re.sub('\\D', '', m.group()))
    @classmethod
    def latest(cls, cache_dir):
        df = cls(cache_dir).load()
        row = df.dropna(subset=['mean_rub', 'median_rub']).iloc[-1]
        return {'year': int(row['year']), 'mean_rub': int(row['mean_rub']), 'median_rub': int(row['median_rub'])}

In [ ]:
from io import StringIO
import pandas as pd
import requests
URL = 'https://raw.githubusercontent.com/selva86/datasets/master/Mall_Customers_Int.csv'
class MallCustomersSource(BaseSource):
    cache_name = 'mall_customers.csv'
    def __init__(self, cache_dir, anchor_rub=62922):
        super().__init__(cache_dir)
        self.anchor_rub = anchor_rub
    def fetch(self):
        text = requests.get(URL, timeout=30, headers={'User-Agent': 'Mozilla/5.0'}).text
        df = pd.read_csv(StringIO(text)).dropna()
        annual_kusd = df['Annual_Income_(k$)'].astype(float)
        monthly_kusd = annual_kusd / 12.0
        scale = self.anchor_rub / monthly_kusd.mean()
        out = pd.DataFrame({'customer_id': df['CustomerID'], 'gender': df['Genre'].map({0: 'Женский', 1: 'Мужской'}), 'age': df['Age'].astype(int), 'monthly_income_rub': (monthly_kusd * scale).round(0).astype(int), 'spending_score': df['Spending_Score'].astype(float)})
        return out.reset_index(drop=True)

In [ ]:
#Цены онлайн-курсов на Stepik
import pandas as pd
import requests
API = 'https://stepik.org/api/courses'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
class StepikCoursesSource(BaseSource):
    cache_name = 'stepik_courses.csv'
    def __init__(self, cache_dir, max_pages=12):
        super().__init__(cache_dir)
        self.max_pages = max_pages
    def fetch(self):
        with requests.Session() as session:
            rows = []
            for page in range(1, self.max_pages + 1):
                params = {'page': page, 'is_paid': 'true', 'language': 'ru', 'is_public': 'true'}
                payload = session.get(API, params=params, timeout=15, headers=HEADERS).json()
                for course in payload.get('courses', []):
                    price = course.get('price')
                    if not course.get('is_paid') or price in (None, '', 0, '0'):
                        continue
                    raw_diff = str(course.get('difficulty') or '').lower()
                    diff_map = {'easy': 'лёгкий', 'normal': 'средний', 'hard': 'сложный'}
                    difficulty = diff_map.get(raw_diff, 'не указан')
                    rows.append({'id': course['id'], 'title': course.get('title', ''), 'price': float(price), 'currency': course.get('currency_code') or 'RUB', 'learners': int(course.get('learners_count') or 0), 'difficulty': difficulty, 'language': course.get('language') or 'ru'})
                if not payload.get('meta', {}).get('has_next'):
                    break
        if not rows:
            raise RuntimeError('Stepik не вернул платных курсов')
        df = pd.DataFrame(rows)
        df = df[df['currency'] == 'RUB'].reset_index(drop=True)
        return df

In [ ]:
"""
Источники:
Dove Global Beauty and Confidence Report (2016, 13 стран, 10500 респондентов): https://www.dove.com/us/en/stories/campaigns/the-dove-global-beauty-and-confidence-report.html
Dove Real Truth About Beauty: Revisited (2010, обновление исследования 2004): https://www.dove.com/uk/stories/about-dove/our-research.html
Dove Self-Esteem Project (исследование подростковой самооценки): https://www.dove.com/us/en/stories/about-dove/dove-self-esteem-project.html
Mental Health Foundation UK, Body Image Report (2019): https://www.mentalhealth.org.uk/our-work/research/body-image-report
NAFI / Аналитический центр НАФИ, исследования потребительского поведения в РФ: https://nafi.ru/analytics/
"""
import pandas as pd
PAINS = [{'Боль': 'Не считают себя красивыми', 'Доля, %': 96.0, 'Источник': 'Dove, Real Truth About Beauty: Revisited (2010)', 'URL': 'https://www.dove.com/uk/stories/about-dove/our-research.html'}, {'Боль': 'Чувствуют давление выглядеть хорошо', 'Доля, %': 71.0, 'Источник': 'Dove Global Beauty and Confidence Report (2016)', 'URL': 'https://www.dove.com/us/en/stories/campaigns/the-dove-global-beauty-and-confidence-report.html'}, {'Боль': 'Стресс и тревога из-за внешности', 'Доля, %': 70.0, 'Источник': 'Mental Health Foundation UK, Body Image Report (2019)', 'URL': 'https://www.mentalhealth.org.uk/our-work/research/body-image-report'}, {'Боль': 'Низкая самооценка у молодёжи из-за внешности', 'Доля, %': 60.0, 'Источник': 'Dove Self-Esteem Project, исследование подростков', 'URL': 'https://www.dove.com/us/en/stories/about-dove/dove-self-esteem-project.html'}, {'Боль': 'Свой главный критик по внешности', 'Доля, %': 54.0, 'Источник': 'Dove Global Beauty and Confidence Report (2016)', 'URL': 'https://www.dove.com/us/en/stories/campaigns/the-dove-global-beauty-and-confidence-report.html'}, {'Боль': 'Избегают активностей из-за внешности', 'Доля, %': 50.0, 'Источник': 'Mental Health Foundation UK, Body Image Report (2019)', 'URL': 'https://www.mentalhealth.org.uk/our-work/research/body-image-report'}]
class ResearchPainsSource(BaseSource):
    cache_name = 'research_pains.csv'
    def fetch(self):
        df = pd.DataFrame(PAINS)
        return df.sort_values('Доля, %', ascending=False).reset_index(drop=True)

## 3. Анализ

In [ ]:
from dataclasses import dataclass
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
FEATURES = ['age', 'monthly_income_rub', 'spending_score']
@dataclass
class Persona:
    name: str
    share: float
    size: int
    avg_age: float
    avg_income: float
    avg_spending: float
    female_share: float
    summary: str = ''
    def __str__(self):
        return self.summary or self.name
    def __repr__(self):
        return f'Persona({self.name!r}, {self.size} чел., средний возраст {self.avg_age:.0f})'
    def as_dict(self):
        return {'Портрет': self.name, 'Доля, %': round(self.share * 100, 1), 'Человек': self.size, 'Возраст': round(self.avg_age, 1), 'Доход, руб/мес': int(round(self.avg_income, 0)), 'Индекс трат': round(self.avg_spending, 1), 'Женщин, %': round(self.female_share * 100, 1)}
class AudienceProfiler:
    def __init__(self, customers, n_clusters=4, seed=42):
        self.customers = customers.copy()
        self.n_clusters = n_clusters
        self.seed = seed
        self._fitted = None
    def fit(self):
        x = StandardScaler().fit_transform(self.customers[FEATURES])
        model = KMeans(n_clusters=self.n_clusters, random_state=self.seed, n_init=10)
        self.customers['cluster'] = model.fit_predict(x)
        self._fitted = self.customers
        return self.customers
    def build(self):
        if self._fitted is None:
            self.fit()
        total = len(self.customers)
        personas = []
        for cluster, group in self.customers.groupby('cluster'):
            p = Persona(name=self._name_for(group), share=len(group) / total, size=len(group), avg_age=group['age'].mean(), avg_income=group['monthly_income_rub'].mean(), avg_spending=group['spending_score'].mean(), female_share=(group['gender'] == 'Женский').mean())
            p.summary = self._describe(p)
            personas.append(p)
        return sorted(personas, key=lambda p: p.avg_spending, reverse=True)
    def _name_for(self, group):
        income = group['monthly_income_rub'].mean()
        spend = group['spending_score'].mean()
        age = group['age'].mean()
        med_income = self.customers['monthly_income_rub'].median()
        med_spend = self.customers['spending_score'].median()
        high_income = income >= med_income
        high_spend = spend >= med_spend
        young = age < self.customers['age'].median()
        if high_spend and (not high_income) and young:
            return 'Активная молодёжь с небольшим доходом'
        if high_income and high_spend:
            return 'Состоятельные и щедрые на себя'
        if high_income and (not high_spend):
            return 'Состоятельные, но экономные'
        if not high_income and (not high_spend):
            return 'Экономные, тратят осторожно'
        return 'Умеренный сегмент'
    @staticmethod
    def _describe(p):
        income = f'{int(p.avg_income):,}'.replace(',', ' ')
        return f'{p.name}: примерно {p.size} человек ({p.share * 100:.0f}% базы), средний возраст {p.avg_age:.0f}, доход около {income} ₽ в месяц, индекс трат {p.avg_spending:.0f} из 100. Женщин {p.female_share * 100:.0f}%.'
    def to_frame(self):
        return pd.DataFrame([p.as_dict() for p in self.build()])
    def afford_segment(self, price_rub, share_of_income=0.5):
        best = None
        best_capacity = -1.0
        for p in self.build():
            capacity = p.avg_income * (p.avg_spending / 100)
            if capacity >= price_rub * share_of_income and capacity > best_capacity:
                best, best_capacity = (p, capacity)
        return best or max(self.build(), key=lambda x: x.avg_income * x.avg_spending)
    def __len__(self):
        return len(self.customers)
    def __iter__(self):
        return iter(self.build())

In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy import stats
@dataclass
class PriceEstimate:
    mean_price: float
    median_price: float
    ci_low: float
    ci_high: float
    recommended_price: float
    coverage: float
    def describe(self):
        return f'Средняя цена курса {self.mean_price:,.0f} руб, медиана {self.median_price:,.0f} руб. С вероятностью 95% настоящее среднее лежит в диапазоне {self.ci_low:,.0f}-{self.ci_high:,.0f} руб. Рекомендуемая цена {self.recommended_price:,.0f} руб попадает в зону {self.coverage * 100:.0f}% покупателей.'.replace(',', ' ')
class WillingnessToPay:
    def __init__(self, courses, price_col='price'):
        self.data = courses
        self.price_col = price_col
    def confidence_interval(self, confidence=0.95):
        x = self.data[self.price_col]
        n = len(x)
        mean = x.mean()
        se = x.std(ddof=1) / np.sqrt(n)
        margin = stats.t.ppf((1 + confidence) / 2, df=n - 1) * se
        return (mean - margin, mean + margin)
    def price_estimate(self, percentile=60):
        x = self.data[self.price_col]
        ci_low, ci_high = self.confidence_interval()
        price = round(float(np.percentile(x, percentile)), 0)
        coverage = (x >= price).mean()
        return PriceEstimate(mean_price=float(x.mean()), median_price=float(x.median()), ci_low=ci_low, ci_high=ci_high, recommended_price=price, coverage=coverage)
    def premium_tier_share(self, threshold_rub):
        x = self.data[self.price_col]
        above = (x >= threshold_rub).mean()
        return {'Порог цены, руб': int(threshold_rub), 'Доля курсов выше порога, %': round(above * 100, 1), '75-й перцентиль, руб': int(x.quantile(0.75)), '90-й перцентиль, руб': int(x.quantile(0.9)), 'Максимум, руб': int(x.max())}
    def by_group(self, col):
        agg = self.data.groupby(col)[self.price_col].agg(['count', 'mean', 'median', 'std']).round(0)
        agg.columns = ['Курсов', 'Средняя цена, руб', 'Медиана, руб', 'Разброс, руб']
        return agg.sort_values('Средняя цена, руб', ascending=False).reset_index().rename(columns={col: 'Группа'})

In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd
from scipy import stats
ALPHA = 0.05
@dataclass
class TestResult:
    name: str
    h0: str
    h1: str
    test: str
    statistic: float
    p_value: float
    reject_h0: bool
    verdict: str
    def as_dict(self):
        return {'Гипотеза': self.name, 'Критерий': self.test, 'Статистика': round(self.statistic, 3), 'p-value': round(self.p_value, 4), 'Отвергаем H0': 'да' if self.reject_h0 else 'нет', 'Вывод': self.verdict}
class HypothesisTester:
    def __init__(self, customers, courses, alpha=ALPHA):
        self.customers = customers
        self.courses = courses
        self.alpha = alpha
    def run_all(self):
        results = [self.income_vs_spending(), self.young_spend_more(), self.popular_costs_more(), self.harder_costs_more()]
        return pd.DataFrame([r.as_dict() for r in results])
    def income_vs_spending(self):
        rho, p = stats.spearmanr(self.customers['monthly_income_rub'], self.customers['spending_score'])
        reject = p < self.alpha
        verdict = f'Связь есть, коэффициент {rho:.2f}' if reject else 'Значимой связи не обнаружено'
        return TestResult('Доход связан с активностью трат', 'Доход и индекс трат независимы', 'Между доходом и индексом трат есть связь', 'Корреляция Спирмена', rho, p, reject, verdict)
    def young_spend_more(self):
        med_age = self.customers['age'].median()
        young = self.customers[self.customers['age'] < med_age]['spending_score']
        older = self.customers[self.customers['age'] >= med_age]['spending_score']
        stat, p = stats.mannwhitneyu(young, older, alternative='greater')
        reject = p < self.alpha
        verdict = 'Молодёжь тратит значимо активнее' if reject else 'Значимой разницы по возрасту не нашли'
        return TestResult('Молодёжь тратит активнее старших', 'Индекс трат у молодых и старших одинаков', 'У молодых индекс трат выше', 'Манна-Уитни (односторонний)', stat, p, reject, verdict)
    def popular_costs_more(self):
        rho, p = stats.spearmanr(self.courses['learners'], self.courses['price'])
        reject = p < self.alpha
        verdict = f'Зависимость есть, коэффициент {rho:.2f}' if reject else 'Цена и популярность не связаны'
        return TestResult('Популярные курсы стоят дороже', 'Цена и количество учеников независимы', 'Цена и количество учеников связаны', 'Корреляция Спирмена (Stepik)', rho, p, reject, verdict)
    def harder_costs_more(self):
        order = {'лёгкий': 1, 'средний': 2, 'сложный': 3}
        data = self.courses[self.courses['difficulty'].isin(order)].copy()
        data['diff_rank'] = data['difficulty'].map(order)
        rho, p = stats.spearmanr(data['diff_rank'], data['price'])
        reject = p < self.alpha
        verdict = f'Связь есть, коэффициент {rho:.2f}' if reject else 'Сложность и цена не связаны'
        return TestResult('Сложные курсы стоят дороже', 'Сложность и цена независимы', 'Чем сложнее курс, тем выше цена', 'Корреляция Спирмена (Stepik)', rho, p, reject, verdict)
    def sample_quality(self):
        x = self.courses['price']
        shapiro_stat, shapiro_p = stats.shapiro(x.sample(min(len(x), 500), random_state=0))
        z = stats.norm.ppf(0.975)
        margin = 0.05 * x.mean()
        needed_n = int(np.ceil((z * x.std(ddof=1) / margin) ** 2))
        return {'Размер выборки (Stepik)': len(x), 'Шапиро-Уилк p-value': round(shapiro_p, 4), 'Распределение нормальное': 'нет' if shapiro_p < self.alpha else 'да', 'Нужный объём для точности 5%': needed_n, 'Объёма достаточно': 'да' if len(x) >= needed_n else 'нет'}

In [ ]:
from dataclasses import dataclass
import pandas as pd
def human(n):
    if abs(n) >= 1000000:
        return f'{n / 1000000:.2f} млн'
    if abs(n) >= 1000:
        return f'{n / 1000:.2f} тыс.'
    return f'{n:.0f}'
@dataclass
class MarketSize:
    tam: int
    sam: int
    som: int
    arpu: float
    som_revenue: float
    assumptions: dict
    def describe(self):
        return f'TAM около {human(self.tam)} человек (парни 18-28 крупных городов в онлайне). SAM около {human(self.sam)} человек (интересна тема внешности). SOM на старте около {human(self.som)} человек. При среднем чеке {self.arpu:,.0f} руб это примерно {human(self.som_revenue)} руб выручки.'.replace(',', ' ')
    def __str__(self):
        return self.describe()
class MarketSizer:
    def __init__(self, cities, wb_latest, courses):
        self.cities = cities
        self.wb = wb_latest
        self.courses = courses
    def estimate(self, top_n_cities=30, youth_share=0.22, age_18_28_in_youth=0.55, male_share=0.46, interest_share=0.4, capture_rate=0.02, arpu_override=None):
        city_pop = int(self.cities.head(top_n_cities)['population'].sum())
        online_share = self.wb.get('Интернет-пользователи, %', {}).get('value', 90.0) / 100
        arpu = float(arpu_override) if arpu_override is not None else float(self.courses['price'].median())
        narrow_share = youth_share * age_18_28_in_youth * male_share
        tam = int(city_pop * narrow_share * online_share)
        sam = int(tam * interest_share)
        som = int(sam * capture_rate)
        return MarketSize(tam=tam, sam=sam, som=som, arpu=arpu, som_revenue=som * arpu, assumptions={'Городов учтено': top_n_cities, 'Население этих городов': city_pop, 'Доля молодёжи 15-34 (Росстат)': youth_share, 'Доля 18-28 внутри 15-34': age_18_28_in_youth, 'Доля мужчин (Росстат)': male_share, 'Доля онлайн (World Bank)': round(online_share, 3), 'Доля с интересом к теме': interest_share, 'Охват на старте': capture_rate, 'Средний чек, ₽': int(arpu)})
    def funnel_frame(self, size):
        people = [size.tam, size.sam, size.som]
        return pd.DataFrame({'Этап': ['TAM (парни 18-28 онлайн)', 'SAM (интересна тема)', 'SOM (охват на старте)'], 'Человек': people, 'Подпись': [human(p) for p in people]})

## 4. Графики и таблицы

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
HEADER_COLOR = TABLE_HEADER
ROW_ALT = TABLE_ROW_ALT
TEXT_DARK = '#222222'
class TableRenderer:
    def __init__(self, figures_dir):
        self.dir = Path(figures_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
    def render(self, df, name, title='', col_width=None):
        df = df.copy()
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].map(self._fmt_number)
        n_rows, n_cols = df.shape
        fig_w = col_width or max(7, 1.7 * n_cols)
        fig, ax = plt.subplots(figsize=(fig_w, 0.5 * n_rows + 1.4))
        ax.axis('off')
        if title:
            ax.set_title(title, fontsize=14, fontweight='bold', pad=16, color=TEXT_DARK)
        table = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)
        for (row, _), cell in table.get_celld().items():
            cell.set_edgecolor('white')
            if row == 0:
                cell.set_facecolor(HEADER_COLOR)
                cell.set_text_props(color='white', fontweight='bold')
            else:
                cell.set_facecolor(ROW_ALT if row % 2 == 0 else 'white')
                cell.set_text_props(color=TEXT_DARK)
        table.auto_set_column_width(col=list(range(n_cols)))
        path = self.dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'таблица сохранена: {name}')
        return path
    @staticmethod
    def _fmt_number(x):
        if pd.isna(x):
            return ''
        if isinstance(x, float) and (not x.is_integer()):
            return f'{x:,.2f}'.replace(',', ' ')
        return f'{int(x):,}'.replace(',', ' ')

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.05)
class Plotter:
    def __init__(self, figures_dir):
        self.dir = Path(figures_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
    def _save(self, fig, name):
        path = self.dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'график сохранён: {name}')
        return path
    def segments_scatter(self, customers):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.scatterplot(data=customers, x='monthly_income_rub', y='spending_score', hue='cluster', palette=PALETTE, s=60, ax=ax)
        ax.set_title('Сегменты покупателей: доход и активность трат')
        ax.set_xlabel('Доход, руб в месяц')
        ax.set_ylabel('Индекс трат (1-100)')
        ax.legend(title='Сегмент')
        return self._save(fig, '01_segments_scatter.png')
    def age_distribution(self, customers):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(data=customers, x='age', hue='gender', multiple='stack', bins=14, palette=GENDER_COLORS, ax=ax)
        ax.set_title('Распределение возраста покупателей')
        ax.set_xlabel('Возраст')
        ax.set_ylabel('Количество')
        return self._save(fig, '02_age_distribution.png')
    def spending_by_segment(self, customers):
        fig, ax = plt.subplots(figsize=(8, 5))
        order = customers.groupby('cluster')['spending_score'].median().sort_values(ascending=False).index
        sns.boxplot(data=customers, x='cluster', y='spending_score', order=order, color=ACCENT_COLOR, ax=ax)
        ax.set_title('Активность трат по сегментам')
        ax.set_xlabel('Сегмент (кластер)')
        ax.set_ylabel('Индекс трат')
        return self._save(fig, '03_spending_by_segment.png')
    def price_distribution(self, courses):
        fig, ax = plt.subplots(figsize=(8, 5))
        cap = courses['price'].quantile(0.95)
        data = courses[courses['price'] <= cap]
        sns.histplot(data=data, x='price', bins=25, color=ACCENT_COLOR, ax=ax)
        ax.set_title('Цены онлайн-курсов в России (Stepik)')
        ax.set_xlabel('Цена курса, руб')
        ax.set_ylabel('Количество курсов')
        return self._save(fig, '04_price_distribution.png')
    def price_by_group(self, courses):
        fig, ax = plt.subplots(figsize=(8, 5))
        data = courses[courses['price'] <= courses['price'].quantile(0.95)].copy()
        order = ['лёгкий', 'средний', 'сложный', 'не указан']
        order = [o for o in order if o in data['difficulty'].unique()]
        sns.boxplot(data=data, x='difficulty', y='price', order=order, color=ACCENT_COLOR, ax=ax)
        ax.set_title('Цена курса по уровню сложности (Stepik)')
        ax.set_xlabel('Уровень сложности')
        ax.set_ylabel('Цена, руб')
        return self._save(fig, '05_price_by_group.png')
    def top_pains(self, pains_df):
        fig, ax = plt.subplots(figsize=(9, 5))
        data = pains_df.sort_values('Доля, %', ascending=True)
        sns.barplot(data=data, y='Боль', x='Доля, %', color=ACCENT_COLOR, ax=ax)
        ax.set_title('Главные боли аудитории (Dove, Mental Health Foundation)')
        ax.set_xlabel('Доля респондентов, %')
        ax.set_ylabel('')
        return self._save(fig, '06_top_pains.png')
    def top_cities(self, cities):
        fig, ax = plt.subplots(figsize=(8, 5))
        data = cities.head(12).iloc[::-1]
        sns.barplot(x=data['population'] / 1000, y=data['city'], color=ACCENT_COLOR, ax=ax)
        ax.set_title('Крупнейшие города России по населению (Wikipedia)')
        ax.set_xlabel('Население, тыс. человек')
        ax.set_ylabel('')
        return self._save(fig, '07_top_cities.png')
    def internet_trend(self, wb):
        fig, ax = plt.subplots(figsize=(8, 5))
        data = wb[wb['indicator'] == 'Интернет-пользователи, %'].sort_values('year')
        sns.lineplot(data=data, x='year', y='value', marker='o', color=ACCENT_COLOR, ax=ax)
        ax.set_title('Охват интернетом в России (World Bank)')
        ax.set_xlabel('Год')
        ax.set_ylabel('Доля населения, %')
        return self._save(fig, '08_internet_trend.png')
    def market_funnel(self, funnel_df):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.barplot(data=funnel_df, x='Человек', y='Этап', color=ACCENT_COLOR, ax=ax)
        labels = funnel_df['Подпись'] if 'Подпись' in funnel_df.columns else funnel_df['Человек']
        for i, (v, label) in enumerate(zip(funnel_df['Человек'], labels)):
            ax.text(v, i, f'  {label}', va='center', fontsize=10)
        ax.set_title('Воронка рынка: TAM / SAM / SOM')
        ax.set_xlabel('Человек')
        ax.set_ylabel('')
        return self._save(fig, '09_market_funnel.png')
    def build_all(self, customers, courses, pains_df):
        return [self.segments_scatter(customers), self.age_distribution(customers), self.spending_by_segment(customers), self.price_distribution(courses), self.price_by_group(courses), self.top_pains(pains_df)]

## 5. Сбор данных

In [ ]:
salary = RosstatSalarySource.latest(DATA_RAW)
print('Якорь дохода: медианная зарплата РФ', salary['median_rub'], 'руб', salary['year'])
customers = MallCustomersSource(DATA_RAW, anchor_rub=salary['median_rub']).load()
courses = StepikCoursesSource(DATA_RAW).load()
cities = WikipediaCitiesParser(DATA_RAW).load()
wb = WorldBankSource(DATA_RAW).load()
wb_latest = WorldBankSource.latest(wb)
pains = ResearchPainsSource(DATA_RAW).load()
customers.head()

## 6. Размер рынка методом TAM-SAM-SOM по цене из анализа конкурентов

In [ ]:
PRICE_RUB = 34820
sizer = MarketSizer(cities, wb_latest, courses)
size = sizer.estimate(arpu_override=PRICE_RUB)
print(size.describe())
size.assumptions

## 7. Портреты ЦА и анализ платёжеспособного сегмента

In [ ]:
profiler = AudienceProfiler(customers)
profiler.fit()
for p in profiler.build():
    print('-', p.summary)
afford = profiler.afford_segment(PRICE_RUB)
print('\nЦенник', PRICE_RUB, 'руб по карману сегменту:', afford.name)
profiler.to_frame()

## 8. Готовность платить: сверка с рынком курсов на Stepik

In [ ]:
wtp = WillingnessToPay(courses)
print(wtp.price_estimate().describe())
wtp.by_group('difficulty')

## 9. Клиентские боли

In [ ]:
pains[['Боль', 'Доля, %', 'Источник']]

## 10. Проверка гипотез и оценка выборки

In [ ]:
tester = HypothesisTester(profiler.customers, courses)
display(tester.run_all())
tester.sample_quality()

## 11. Графики

In [ ]:
from IPython.display import Image, display
plotter = Plotter(FIGURES)
funnel = sizer.funnel_frame(size)
paths = [plotter.market_funnel(funnel), plotter.top_cities(cities), plotter.internet_trend(wb), *plotter.build_all(profiler.customers, courses, pains)]
for pth in paths:
    display(Image(str(pth)))

## 12. Дэшборд

In [ ]:
"""
http://127.0.0.1:8050
"""
import sys
from pathlib import Path
import plotly.express as px
from dash import Dash, dash_table, dcc, html
from dash.dependencies import Input, Output
SALARY = RosstatSalarySource.latest(DATA_RAW)
CUSTOMERS = AudienceProfiler(MallCustomersSource(DATA_RAW, anchor_rub=SALARY['median_rub']).load()).fit()
COURSES = StepikCoursesSource(DATA_RAW).load()
CITIES = WikipediaCitiesParser(DATA_RAW).load()
WB = WorldBankSource(DATA_RAW).load()
WB_LATEST = WorldBankSource.latest(WB)
PRICE_RUB = 34820
MARKET = MarketSizer(CITIES, WB_LATEST, COURSES).estimate(arpu_override=PRICE_RUB)
_PROFILER = AudienceProfiler(MallCustomersSource(DATA_RAW, anchor_rub=SALARY['median_rub']).load())
_PROFILER.fit()
AFFORD = _PROFILER.afford_segment(PRICE_RUB)
PAINS = ResearchPainsSource(DATA_RAW).load()
HYP = HypothesisTester(CUSTOMERS, COURSES).run_all()
COLORWAY = SEQ_COLORS
app = Dash(__name__, title='Анализ ЦА')
server = app.server
def card(title, value):
    return html.Div(style={'flex': '1', 'background': TABLE_ROW_ALT, 'borderRadius': '12px', 'padding': '16px'}, children=[html.Div(title, style={'fontSize': '13px', 'color': '#555'}), html.Div(value, style={'fontSize': '24px', 'fontWeight': 'bold'})])
def market_funnel_fig():
    funnel = MarketSizer(CITIES, WB_LATEST, COURSES).funnel_frame(MARKET)
    fig = px.funnel(funnel, x='Человек', y='Этап', text='Подпись', title='Воронка рынка: TAM / SAM / SOM', color_discrete_sequence=[ACCENT_COLOR])
    return fig
def top_cities_fig():
    data = CITIES.head(12)
    fig = px.bar(data, x='population', y='city', orientation='h', title=f"Крупнейшие города России, {int(CITIES['year'].iloc[0])} (Wikipedia)", color_discrete_sequence=[ACCENT_COLOR])
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, xaxis_title='Население', yaxis_title='')
    return fig
def internet_fig():
    data = WB[WB['indicator'] == 'Интернет-пользователи, %'].sort_values('year')
    fig = px.line(data, x='year', y='value', markers=True, title='Охват интернетом в России (World Bank)', color_discrete_sequence=[ACCENT_COLOR])
    fig.update_layout(xaxis_title='Год', yaxis_title='Доля населения, %')
    return fig
def pains_fig():
    data = PAINS.sort_values('Доля, %')
    fig = px.bar(data, x='Доля, %', y='Боль', orientation='h', title='Главные боли аудитории (Dove, Mental Health Foundation)', hover_data={'Источник': True}, color_discrete_sequence=[ACCENT_COLOR])
    fig.update_layout(yaxis_title='')
    return fig
app.layout = html.Div(style={'maxWidth': '1100px', 'margin': '0 auto', 'fontFamily': 'Inter, Arial, sans-serif', 'padding': '24px'}, children=[html.H1('GlowUp: целевая аудитория'), html.P('Кому продаём, какие боли клиента и сколько они готовы платить. Данные спарсены из пабликов ВК и с сайта hh.ru.'), html.Div(style={'display': 'flex', 'gap': '12px', 'alignItems': 'center', 'margin': '16px 0'}, children=[html.Label('Пол:'), dcc.Dropdown(id='gender', options=[{'label': g, 'value': g} for g in ['Все', 'Женский', 'Мужской']], value='Все', clearable=False, style={'width': '260px'})]), html.Div(id='cards', style={'display': 'flex', 'gap': '16px', 'marginBottom': '8px'}), dcc.Tabs([dcc.Tab(label='Рынок', children=[html.Br(), html.P('Размер рынка: население городов из Wikipedia, охват интернетом из World Bank, средний чек из реальных продаж бьюти-товаров.'), dcc.Graph(figure=market_funnel_fig()), dcc.Graph(figure=top_cities_fig()), dcc.Graph(figure=internet_fig())]), dcc.Tab(label='Сегменты и портреты', children=[dcc.Graph(id='segments'), dcc.Graph(id='age')]), dcc.Tab(label='Платёжеспособность', children=[dcc.Graph(id='price_dist'), dcc.Graph(id='price_group')]), dcc.Tab(label='Боли клиента', children=[dcc.Graph(figure=pains_fig())]), dcc.Tab(label='Гипотезы', children=[html.Br(), dash_table.DataTable(data=HYP.to_dict('records'), columns=[{'name': c, 'id': c} for c in HYP.columns], style_cell={'textAlign': 'left', 'fontFamily': 'Arial', 'padding': '8px', 'whiteSpace': 'normal'}, style_header={'fontWeight': 'bold', 'backgroundColor': TABLE_ROW_ALT})])])])
def filtered_customers(gender):
    return CUSTOMERS if gender == 'Все' else CUSTOMERS[CUSTOMERS['gender'] == gender]
@app.callback(Output('cards', 'children'), Output('segments', 'figure'), Output('age', 'figure'), Output('price_dist', 'figure'), Output('price_group', 'figure'), Input('gender', 'value'))
def update(gender):
    cust = filtered_customers(gender)
    wtp = WillingnessToPay(COURSES)
    est = wtp.price_estimate()
    cards = [card('Цена GlowUp', f'{PRICE_RUB:,} ₽'.replace(',', ' ')), card('Платёжеспособный сегмент', AFFORD.name), card('Доход сегмента', f'{int(AFFORD.avg_income):,} ₽/мес'.replace(',', ' ')), card('SOM (потолок)', f'{MARKET.som:,} чел'.replace(',', ' '))]
    seg_fig = px.scatter(cust, x='monthly_income_rub', y='spending_score', color='cluster', title='Сегменты: доход и активность трат', color_continuous_scale=COLORWAY, labels={'monthly_income_rub': 'Доход, ₽/мес', 'spending_score': 'Индекс трат'})
    age_fig = px.histogram(cust, x='age', color='gender', nbins=14, title='Возраст покупателей', color_discrete_map=GENDER_COLORS)
    age_fig.update_layout(bargap=0.05)
    cap = COURSES['price'].quantile(0.95)
    courses_clean = COURSES[COURSES['price'] <= cap]
    price_dist = px.histogram(courses_clean, x='price', nbins=25, title='Цены онлайн-курсов в России (Stepik)', color_discrete_sequence=[ACCENT_COLOR])
    price_dist.update_layout(xaxis_title='Цена курса, руб', yaxis_title='Курсов')
    price_group = px.box(courses_clean, x='difficulty', y='price', title='Цена курса по уровню сложности', color_discrete_sequence=[ACCENT_COLOR])
    price_group.update_layout(xaxis_title='Сложность', yaxis_title='Цена, руб')
    return (cards, seg_fig, age_fig, price_dist, price_group)